# MLE-06 · Unsupervised Learning Foundations
### Classical ML completion — structure without labelled targets

**Prerequisites:** FND-01, FND-02, MLE-02, MLE-03, and EVAL-01.
**Estimated time:** 6–8 hours.
Supervised learning asks whether predictions match labels. Here labels are absent,
so assumptions, stability, geometry, and downstream usefulness become central.


> **Canonical learner route · module MLE-06 of 65**
>
> Required prerequisites: **FND-01, FND-02, MLE-02, MLE-03, EVAL-01** · Previous: **EVAL-01** ·
> Next after mastery: **DL-01** · Expected first-pass workload:
> **5–8 hours**
>
> **Core path:** intuition, objectives, foundations, runnable implementation,
> failure modes, and assessed exercises. **Extension path:** history, production,
> tradeoffs, and interview material may be revisited after the core pass.
> Do not continue merely because every cell ran. Continue when you can complete
> the independent exercise and teach-back without notes. The canonical route in
> `docs/CURRICULUM_PATH.json` is authoritative when section-local file order and
> prerequisite order differ.


## 1 · Learning Objectives

Standardize data safely; use PCA/SVD for lower-dimensional representation;
explain variance retained and reconstruction error; implement k-means; distinguish
clustering from classification; evaluate clusters without pretending they are
ground truth; and use anomaly scores with explicit review thresholds.


<details>
<summary><strong>Mathematics notation support — open when a formula feels dense</strong></summary>

- $x_i$: item or component number $i$; a subscript is a label, not multiplication.
- $n$: number of examples; $d$: number of features or dimensions.
- $\mathbf x$: vector; $X$: matrix; $X^\top$: transpose (rows and columns swap).
- $\hat y$: an estimate or prediction; a hat marks an estimated quantity.
- $\sum$: add repeated terms; $\prod$: multiply repeated terms.
- $\lVert\mathbf x\rVert$: vector length; $|x|$: distance of one number from zero.
- $\frac{\partial f}{\partial x}$: slope of $f$ as $x$ changes while other inputs
  stay fixed; $\nabla f$: vector containing all parameter slopes.
- $P(A\mid B)$: probability of $A$ after restricting attention to cases where
  $B$ is true.
- $\log$ reverses an exponential and turns products into sums.

Read a formula one operator at a time, write object shapes beside vectors and
matrices, and substitute a tiny numeric example. Review PRE-01 through PRE-04 for
worked explanations of these symbols.
</details>


## Student Lesson Companion · MLE-06 · Unsupervised Learning Foundations

Use this companion during the **core pass**. Write short answers before reading
the extension material; then correct them in a different colour after the lesson.

### Practical problem before history

1. What concrete task or decision are we trying to improve?
2. Why is it difficult with the data, time, labels, or compute available?
3. What is the simplest previous baseline?
4. Where does that baseline fail?
5. What capability must the new concept add?

Section 2 must supply enough evidence to answer these questions. Historical detail
is extension material unless it explains the problem or design constraint.

### Concept, analogy, and analogy limit

After Section 3, explain the concept in one sentence without unexplained jargon.
Map it to one daily-life analogy **or** one concrete visual example. Then state
one place where the mapping breaks; an analogy is a bridge, not a proof.

### Use / avoid / alternatives

Complete this decision table from Sections 7, 9, and 11:

| Decision | Required answer |
|---|---|
| Use it when | Three realistic conditions where its assumptions and benefits fit |
| Avoid it when | Two conditions where it is misleading, unsafe, or wasteful |
| Prefer instead | At least one simpler baseline and one alternative for a failed assumption |
| Evidence | Metric, diagnostic, or constraint that supports the choice |

### Code walkthrough and expected-result contract

For the first implementation, annotate: inputs → shapes/units → initialization →
central computation → intermediate output → final output → verification. Before
execution, record the expected value, range, or shape and what it would mean.

Distinguish these outcomes:

| Outcome | Interpretation | Next action |
|---|---|---|
| Exception, non-finite value, impossible shape | Code or data-contract failure | Inspect the first violated boundary |
| Output has valid type/shape but weak metric | Experiment ran; method may be poor | Diagnose data, assumptions, and baseline comparison |
| Strong metric on training data only | Insufficient evidence | Evaluate with the declared validation design |
| Expected output on untouched data | Supports one scoped claim | Record limitations; do not generalize beyond evidence |

### Debugging table

| Symptom | Likely cause | Inspect | Scoped fix |
|---|---|---|---|
| Shape/type error | Interface mismatch | Shapes, dtypes, feature names | Repair the boundary, not downstream symptoms |
| NaN/Inf or divergence | Invalid input or unstable update | Raw values, loss, gradients, learning rate | Clean/validate input or change one optimization control |
| Implausibly strong result | Leakage or invalid split | Fit boundaries, timestamps, duplicate entities | Rebuild the evaluation path before tuning |
| Different repeated result | Uncontrolled state | Seeds, data order, train/eval mode, versions | Record and control randomness intentionally |
| Plausible output but poor result | Wrong assumptions or representation | Baseline, slices, residuals/errors | Prefer a justified alternative; do not debug valid code as broken |


## 2 · Historical Motivation

Many datasets contain measurements but no reliable labels. Dimensionality
reduction summarizes correlated variables, clustering proposes groups for
investigation, and anomaly detection prioritizes unusual cases. These techniques
are exploratory tools, not automatic discoveries of real categories.


## 3 · Intuition and Visual Understanding

PCA rotates the coordinate system toward directions with the most variation.
K-means alternates between assigning points to the nearest center and moving each
center to its assigned mean. Anomaly detection ranks unusualness; a threshold
turns ranking into an operational review decision.


## 4 · Mathematical Foundations

PCA chooses a unit vector $v$ maximizing projected variance:
$$v_1=\arg\max_{\lVert v\rVert=1}\operatorname{Var}(Xv).$$
$X\in\mathbb R^{n\times d}$ is centered data and $Xv$ is one score per row.
For two perfectly correlated standardized features, the first direction is
proportional to `(1,1)` and captures nearly all variance. PCA is scale-sensitive
and high variance does not necessarily mean high task value.

K-means minimizes $\sum_i\lVert x_i-\mu_{c_i}\rVert^2$, where $c_i$ assigns row
$i$ to center $\mu$. It prefers roughly spherical clusters under Euclidean distance.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.ensemble import IsolationForest

data = load_wine()
X = StandardScaler().fit_transform(data.data)
projection = PCA(n_components=2, random_state=42).fit_transform(X)
print("shape", X.shape, "projected", projection.shape)


## 5 · Manual Implementation from Scratch

K-means needs an initialization, assignment step, update step, and stopping rule.
Run multiple initializations because the objective is non-convex.


In [ ]:
def kmeans(X, k, seed=42, max_iter=100):
    rng = np.random.default_rng(seed)
    centers = X[rng.choice(len(X), size=k, replace=False)].copy()
    for _ in range(max_iter):
        labels = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2).argmin(axis=1)
        updated = np.vstack([X[labels == j].mean(axis=0) for j in range(k)])
        if np.allclose(updated, centers):
            break
        centers = updated
    return labels, centers

labels, centers = kmeans(projection, 3)
assert labels.shape == (len(X),)


## 6 · Visualization

A two-dimensional PCA plot is a projection, not proof that clusters exist in the
original space. Use it to form questions, then test stability and domain meaning.


In [ ]:
plt.scatter(projection[:, 0], projection[:, 1], c=labels, cmap="tab10", s=25)
plt.xlabel("principal component 1"); plt.ylabel("principal component 2")
plt.title("K-means assignments shown in a PCA projection")
plt.tight_layout(); plt.show()


## 7 · Failure Modes and Common Mistakes

- Fitting scaling/PCA on all rows before an evaluated downstream split.
- Calling a cluster a customer “type” without external evidence.
- Selecting k only because one plot looks attractive.
- Using Euclidean distance on incomparable or categorical features.
- Treating anomaly score as fraud probability.
- Ignoring instability across seeds and samples.


## 8 · Library Implementation

Compare k values using inertia, silhouette, stability, and usefulness—not one
metric alone. Preserve the preprocessing and fitted representation together.


In [ ]:
for k in range(2, 6):
    candidate = KMeans(n_clusters=k, n_init=20, random_state=42).fit(X)
    print(k, round(candidate.inertia_, 1), round(silhouette_score(X, candidate.labels_), 3))
detector = IsolationForest(contamination=0.05, random_state=42).fit(X)
scores = -detector.score_samples(X)
print("top anomaly row indices", np.argsort(scores)[-5:][::-1])


## 9 · Realistic Case Study

A laboratory uses PCA to identify redundant measurements, clustering to propose
review cohorts, and anomaly scores to prioritize instrument-quality checks. Domain
experts review all interpretations; no cluster label becomes a decision target
without validation.


## 10 · Production and Learning Considerations

Version preprocessing, component loadings, centers, thresholds, and feature
schema. Monitor reconstruction error, assignment distance, cluster-size drift,
and review yield. Refit only after checking that changed structure is real.


## 11 · Tradeoff Analysis

PCA is efficient and interpretable through loadings but linear. K-means scales
well but assumes Euclidean geometry. Density methods find irregular clusters but
are sensitive to scale and density. Isolation Forest ranks anomalies efficiently
but does not explain their operational significance.


## 12 · Readiness and Interview Preparation

Explain why clustering has no ordinary accuracy, why scaling changes k-means,
how PCA differs from feature selection, and how you would determine whether an
anomaly detector saves reviewer time.


## 13 · Teach-Back

Explain PCA without saying “it compresses data,” then explain why a colorful
cluster plot is not evidence of natural categories. Give one safe and one unsafe
operational use of anomaly scores.


## 14 · Exercises, Self-Check, and Solutions

**Worked:** standardize two differently scaled features and compare distances.
**Guided (30 min):** reconstruct standardized wine data from 2, 5, and 10 PCA
components; plot reconstruction error. **Self-check:** error cannot increase as
components are added. **Independent (45 min):** compare k=2…6 across five seeds
using silhouette and assignment stability. **Challenge (60 min):** set an anomaly
review threshold from a fixed review budget and report review yield assumptions.

<details><summary><strong>Solution and scoring rubric</strong></summary>
Use `inverse_transform` for reconstruction; report mean squared reconstruction
error. Stability requires matching or pairwise co-assignment rather than comparing
arbitrary cluster numbers. Award 3 points for preprocessing/reconstruction, 3 for
multi-seed evaluation, 2 for threshold reasoning, and 2 for limitations.
Common mistakes: interpreting cluster IDs as ordered labels and fitting on future
evaluation data. **Readiness threshold: 8/10.**
</details>


## Lesson Close · Summary, Student Check, and Memory Aid

### Five short student checks

1. What practical problem does **MLE-06 · Unsupervised Learning Foundations** solve?
2. What is its central mechanism in simple language?
3. Which assumption or limitation is easiest to forget?
4. What output or diagnostic tells you it worked as intended?
5. When would you choose a simpler or related alternative?

### Plain-language summary

Complete four sentences without notes: **The problem is… The concept works by…
I would use it when… I would avoid it when…** Compare your answer with the
objectives, failure modes, tradeoff analysis, and teach-back section.

### One-sentence memory aid

**Remember MLE-06 · Unsupervised Learning Foundations: start from the problem, trace the mechanism, verify the
evidence, and use it only when its assumptions fit.** Replace this general aid
with your own topic-specific sentence of no more than 20 words.

The lesson is complete only after the Required Core Mastery Gate, not after the
final code cell runs.


## Required Core Mastery Gate · MLE-06

Complete this gate before treating the module as finished. The longer exercises
in Section 14 are extension practice unless the module says otherwise.

**Worked example:** rerun the smallest worked example in this notebook. Annotate
every input, output, shape or unit, and the claim the result supports.

**Guided practice (20–30 min):** change one input in that example. Before running
it, predict the direction of change and explain why. Use the nearest preceding
formula or algorithm step as a hint. **Self-check:** compare prediction with the
result and explain any mismatch rather than editing the prediction afterward.

**Independent practice (30–45 min):** on fresh tiny data, reproduce the module's
central operation without copying the completed cell. State assumptions, expected
output, and one invalid input. **Self-check:** verify with an assertion, a manual
calculation, or a trusted library implementation.

**Challenge (30–60 min):** create one failure described in Section 7, detect it
using evidence, and repair it without changing unrelated variables.

**Solution and scoring rubric:** 2 points for a correct prediction, 2 for a
runnable independent implementation, 2 for an objective self-check, 2 for failure
diagnosis, and 2 for teach-back without notes. Common mistakes: copying before
attempting, using later-module concepts as unexplained shortcuts, evaluating on
training data, and continuing because cells ran. **Readiness threshold: 8/10**,
including both independent implementation and teach-back points. If below 8,
revisit the named prerequisite in the route card and retry with different data.
